# NAICS Crosswalk Rebuild

This notebook rebuilds the NAICS Felten crosswalk in a notebook-native workflow.

We keep the process in distinct stages:

1. Raw Appendix B join baseline
2. Accepted suggested joins
3. Combined notebook crosswalk
4. Manual review and notebook locks
5. Final canonical NAICS join table


## 1. Setup

Load packages and define repo-relative paths used throughout the rebuild.


In [10]:
from __future__ import annotations

from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)

NOTEBOOK_ROOT = Path.cwd()
REPO_ROOT = NOTEBOOK_ROOT
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent

DB_PATH = REPO_ROOT / "foundations" / "etl" / "data" / "duckdb" / "patterns_in_place.duckdb"
FELTEN_WORKBOOK_PATH = REPO_ROOT / "metro-deep-dive" / "metro-area-explorer" / "industry" / "reference_data" / "AIOE_DataAppendix.xlsx"
AUDIT_ROOT = REPO_ROOT / "metro-deep-dive" / "metro-area-explorer" / "industry" / "outputs" / "national" / "d6_coverage_review"
NAICS_RECOMMENDATIONS_PATH = AUDIT_ROOT / "recommended_felten_naics_overrides_initial.csv"
NAICS_LOCKED_OVERRIDES_PATH = AUDIT_ROOT / "locked_felten_naics_manual_overrides_national_2024.csv"
NAICS_STEP4_REVIEW_BASE_PATH = REPO_ROOT / "metro-deep-dive" / "analysis_program" / "01_ai_inversion" / "outputs" / "naics_crosswalk_step4_review_base.csv"
NAICS_FINAL_CROSSWALK_OUTPUT_PATH = REPO_ROOT / "metro-deep-dive" / "analysis_program" / "01_ai_inversion" / "outputs" / "naics_felten_join_reference.csv"

display(pd.DataFrame([
    {"path": "DuckDB", "value": str(DB_PATH.relative_to(REPO_ROOT))},
    {"path": "Felten workbook", "value": str(FELTEN_WORKBOOK_PATH.relative_to(REPO_ROOT))},
    {"path": "Accepted suggested joins", "value": str(NAICS_RECOMMENDATIONS_PATH.relative_to(REPO_ROOT))},
    {"path": "Locked manual overrides", "value": str(NAICS_LOCKED_OVERRIDES_PATH.relative_to(REPO_ROOT))},
    {"path": "Step 4 review base output", "value": str(NAICS_STEP4_REVIEW_BASE_PATH.relative_to(REPO_ROOT))},
]))


,path,value
0,DuckDB,foundations/etl/data/duckdb/patterns_in_place....
1,Felten workbook,metro-deep-dive/metro-area-explorer/industry/r...
2,Accepted suggested joins,metro-deep-dive/metro-area-explorer/industry/o...
3,Locked manual overrides,metro-deep-dive/metro-area-explorer/industry/o...
4,Step 4 review base output,metro-deep-dive/analysis_program/01_ai_inversi...


## 2. Raw Appendix B Baseline

Start from the raw Felten Appendix B table and compare it to the live 4-digit NAICS surface.


In [11]:
# Read the raw Felten Appendix B table.
# This is the static source table before we apply any reviewed suggestions or manual overrides.
appendix_b = pd.read_excel(FELTEN_WORKBOOK_PATH, sheet_name="Appendix B").rename(
    columns={
        "NAICS": "felten_naics_code",
        "Industry Title": "felten_naics_title",
        "AIIE": "aiie_score",
    }
).copy()
appendix_b["felten_naics_code"] = appendix_b["felten_naics_code"].astype(str).str.strip()
appendix_b["felten_naics_title"] = appendix_b["felten_naics_title"].astype(str).str.strip()
appendix_b["aiie_score"] = pd.to_numeric(appendix_b["aiie_score"], errors="coerce")

display(appendix_b.head(20))
display(pd.DataFrame([
    {"metric": "Appendix B rows", "value": len(appendix_b)},
    {"metric": "Appendix B distinct codes", "value": appendix_b["felten_naics_code"].nunique()},
    {"metric": "Appendix B rows with missing score", "value": int(appendix_b["aiie_score"].isna().sum())},
]))


,felten_naics_code,felten_naics_title,aiie_score
0,1133,Logging,-1.360161
1,1151,Support Activities for Crop Production,-2.165304
2,1152,Support Activities for Animal Production,-1.149307
3,2111,Oil and Gas Extraction,0.668615
4,2121,Coal Mining,-1.146242
5,2122,Metal Ore Mining,-0.892403
6,2123,Nonmetallic Mineral Mining and Quarrying,-1.037903
7,2131,Support Activities for Mining,-0.981041
8,2211,"Electric Power Generation, Transmission and Di...",0.209619
9,2212,Natural Gas Distribution,0.345354


,metric,value
0,Appendix B rows,250
1,Appendix B distinct codes,246
2,Appendix B rows with missing score,0


In [12]:
# Pull the live yearly 4-digit NAICS surface from QCEW.
# We keep both yearly and weighted employment fields so we can inspect raw coverage before any review decisions.
naics_reference_sql = """
WITH naics_base AS (
    SELECT
        c.period AS year,
        c.industry_code AS naics_code,
        any_value(c.industry_title) AS naics_title,
        SUM(c.annual_avg_emplvl) AS sector_employment
    FROM staging.bls_qcew_county c
    INNER JOIN silver.bls_qcew_industry_map m
        ON c.industry_code = m.industry_code
    WHERE c.own_code = '5'
      AND m.code_type = 'naics_industry_group'
    GROUP BY 1, 2
)
SELECT
    year,
    naics_code,
    naics_title,
    sector_employment,
    SUM(sector_employment) OVER (PARTITION BY year) AS total_employment,
    sector_employment / SUM(sector_employment) OVER (PARTITION BY year) AS sector_weight
FROM naics_base
ORDER BY year, sector_employment DESC, naics_code
"""

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    naics_reference_yearly = con.execute(naics_reference_sql).fetchdf()

naics_reference_codes = (
    naics_reference_yearly.sort_values(["year", "sector_employment"], ascending=[False, False])
    .drop_duplicates(subset=["naics_code"])[["naics_code", "naics_title"]]
    .rename(columns={"naics_title": "naics_title_ours"})
    .reset_index(drop=True)
)

display(naics_reference_yearly.head(20))
display(pd.DataFrame([
    {"metric": "Live NAICS rows", "value": len(naics_reference_yearly)},
    {"metric": "Live distinct NAICS codes", "value": naics_reference_yearly["naics_code"].nunique()},
    {"metric": "First year", "value": int(naics_reference_yearly["year"].min())},
    {"metric": "Latest year", "value": int(naics_reference_yearly["year"].max())},
]))


,year,naics_code,naics_title,sector_employment,total_employment,sector_weight
0,2010,7221,Full-service restaurants,4409953.0,91966397.0,0.047952
1,2010,7222,Limited-service eating places,4016936.0,91966397.0,0.043678
2,2010,5613,Employment services,2649886.0,91966397.0,0.028814
3,2010,6221,General medical and surgical hospitals,2433910.0,91966397.0,0.026465
4,2010,4451,Grocery stores,2346997.0,91966397.0,0.025520
5,2010,6211,Offices of physicians,2287112.0,91966397.0,0.024869
6,2010,5511,Management of companies and enterprises,1772086.0,91966397.0,0.019269
7,2010,5617,Services to buildings and dwellings,1730923.0,91966397.0,0.018821
8,2010,5221,Depository credit intermediation,1657004.0,91966397.0,0.018017
9,2010,2382,Building equipment contractors,1623282.0,91966397.0,0.017651


,metric,value
0,Live NAICS rows,4559
1,Live distinct NAICS codes,332
2,First year,2010
3,Latest year,2024


In [13]:
# Build the raw NAICS join in two passes.
# First we measure strict exact-code coverage against Appendix B.
# Then we apply a small, explicit normalization map for known NAICS vintage / aggregate differences
# so we can see how much of the raw gap is really a code-system reconciliation issue.

appendix_b_code_lookup = (
    appendix_b.sort_values(["felten_naics_code", "felten_naics_title"], kind="mergesort")
    .drop_duplicates(subset=["felten_naics_code"], keep="first")
    .reset_index(drop=True)
)

appendix_b_duplicate_code_review = (
    appendix_b.groupby("felten_naics_code", as_index=False)
    .agg(
        row_count=("felten_naics_code", "size"),
        title_count=("felten_naics_title", "nunique"),
        score_count=("aiie_score", lambda values: int(values.notna().sum())),
    )
    .loc[lambda df: df["row_count"] > 1]
    .sort_values(["row_count", "felten_naics_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

# This map is a raw code-normalization step, not a review override.
# It handles known live-to-Felten code vintage differences before we move into accepted suggestions.
NAICS_RAW_NORMALIZATION_MAP = {
    "4451": "4450",
    "4452": "4450",
    "4492": "4431",
    "4551": "4520",
    "4552": "4520",
    "4561": "4461",
    "4571": "4471",
    "4581": "4481",
    "4582": "4482",
    "4583": "4483",
    "4591": "4511",
    "4592": "4512",
    "4593": "4531",
    "4594": "4532",
    "4595": "4533",
    "4599": "4539",
    "5132": "5112",
    "5162": "5152",
    "5171": "5170",
    "5192": "5191",
    "5221": "5220",
    "5222": "5220",
    "5223": "5220",
    "5239": "5230",
    "5311": "5310",
    "5313": "5310",
    "5324": "5320",
}

def summarize_naics_coverage(df: pd.DataFrame, flag_col: str) -> pd.DataFrame:
    yearly = (
        df.groupby("year", as_index=False)
        .agg(
            total_codes=("naics_code", "size"),
            matched_codes=(flag_col, "sum"),
            total_weight=("sector_weight", "sum"),
            matched_weight=("sector_weight", lambda values: float(values[df.loc[values.index, flag_col]].sum())),
        )
    )
    yearly["coverage_pct"] = yearly["matched_codes"] / yearly["total_codes"]
    yearly["weighted_coverage_pct"] = yearly["matched_weight"] / yearly["total_weight"]

    overall = pd.DataFrame([
        {
            "year": "all_years",
            "total_codes": len(df),
            "matched_codes": int(df[flag_col].sum()),
            "total_weight": float(df["sector_weight"].sum()),
            "matched_weight": float(df.loc[df[flag_col], "sector_weight"].sum()),
        }
    ])
    overall["coverage_pct"] = overall["matched_codes"] / overall["total_codes"]
    overall["weighted_coverage_pct"] = overall["matched_weight"] / overall["total_weight"]

    return pd.concat([yearly, overall], ignore_index=True)[["year", "coverage_pct", "weighted_coverage_pct"]]

naics_raw_status = (
    naics_reference_yearly[["year", "naics_code", "naics_title", "sector_employment", "total_employment", "sector_weight"]]
    .drop_duplicates()
    .merge(
        naics_reference_codes,
        on="naics_code",
        how="left",
        validate="many_to_one",
    )
)

naics_raw_status = naics_raw_status.merge(
    appendix_b_code_lookup.rename(
        columns={
            "felten_naics_code": "felten_naics_code_exact",
            "felten_naics_title": "felten_naics_title_exact",
            "aiie_score": "aiie_score_exact",
        }
    ),
    left_on="naics_code",
    right_on="felten_naics_code_exact",
    how="left",
    validate="many_to_one",
)

naics_raw_status["normalized_naics_code"] = naics_raw_status["naics_code"].map(NAICS_RAW_NORMALIZATION_MAP).fillna(naics_raw_status["naics_code"])
naics_raw_status = naics_raw_status.merge(
    appendix_b_code_lookup.rename(
        columns={
            "felten_naics_code": "felten_naics_code_normalized",
            "felten_naics_title": "felten_naics_title_normalized",
            "aiie_score": "aiie_score_normalized",
        }
    ),
    left_on="normalized_naics_code",
    right_on="felten_naics_code_normalized",
    how="left",
    validate="many_to_one",
)

naics_raw_status["raw_exact_match_flag"] = naics_raw_status["aiie_score_exact"].notna()
naics_raw_status["raw_normalized_match_flag"] = naics_raw_status["aiie_score_normalized"].notna()
naics_raw_status["normalization_applied_flag"] = naics_raw_status["normalized_naics_code"] != naics_raw_status["naics_code"]
naics_raw_status["normalization_created_match_flag"] = (~naics_raw_status["raw_exact_match_flag"]) & naics_raw_status["raw_normalized_match_flag"]

naics_raw_exact_coverage = summarize_naics_coverage(naics_raw_status, "raw_exact_match_flag")
naics_raw_normalized_coverage = summarize_naics_coverage(naics_raw_status, "raw_normalized_match_flag")

latest_year = int(naics_reference_yearly["year"].max())
naics_raw_missing_latest = (
    naics_raw_status.loc[
        (naics_raw_status["year"] == latest_year) & (~naics_raw_status["raw_normalized_match_flag"]),
        ["year", "naics_code", "naics_title", "sector_weight"],
    ]
    .sort_values(["sector_weight", "naics_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

naics_raw_normalization_effect = (
    naics_raw_status.loc[
        naics_raw_status["normalization_created_match_flag"],
        [
            "year",
            "naics_code",
            "naics_title",
            "normalized_naics_code",
            "felten_naics_title_normalized",
            "aiie_score_normalized",
            "sector_weight",
        ],
    ]
    .sort_values(["year", "sector_weight", "naics_code"], ascending=[False, False, True], kind="mergesort")
    .reset_index(drop=True)
)

display(pd.DataFrame([
    {"metric": "Appendix B duplicate code groups", "value": len(appendix_b_duplicate_code_review)},
    {"metric": "Latest-year unresolved after raw normalization", "value": len(naics_raw_missing_latest)},
    {"metric": "Rows helped by raw normalization", "value": int(naics_raw_status["normalization_created_match_flag"].sum())},
]))
display(appendix_b_duplicate_code_review)
display(naics_raw_exact_coverage.style.format({"coverage_pct": "{:.1%}", "weighted_coverage_pct": "{:.1%}"}))
display(naics_raw_normalized_coverage.style.format({"coverage_pct": "{:.1%}", "weighted_coverage_pct": "{:.1%}"}))
display(naics_raw_normalization_effect.head(30))
display(naics_raw_missing_latest.head(50))


,metric,value
0,Appendix B duplicate code groups,3
1,Latest-year unresolved after raw normalization,70
2,Rows helped by raw normalization,175


,felten_naics_code,row_count,title_count,score_count
0,4240,3,3,3
1,3250,2,2,2
2,3320,2,2,2


,year,coverage_pct,weighted_coverage_pct
0,2010,73.9%,72.1%
1,2011,74.4%,81.4%
2,2012,74.4%,81.5%
3,2013,74.4%,82.1%
4,2014,74.4%,82.3%
5,2015,74.4%,82.5%
6,2016,74.4%,82.7%
7,2017,74.7%,83.0%
8,2018,74.7%,83.1%
9,2019,74.7%,83.4%


,year,coverage_pct,weighted_coverage_pct
0,2010,76.8%,79.1%
1,2011,77.4%,88.2%
2,2012,77.4%,88.4%
3,2013,77.4%,88.9%
4,2014,77.4%,89.0%
5,2015,77.4%,89.1%
6,2016,77.4%,89.3%
7,2017,77.3%,89.1%
8,2018,77.3%,89.1%
9,2019,77.3%,89.3%


,year,naics_code,naics_title,normalized_naics_code,felten_naics_title_normalized,aiie_score_normalized,sector_weight
0,2024,4451,Grocery stores,4450,Food and Beverage Stores (4451 and 4452 only),-0.406696,0.023043
1,2024,4552,"NAICS 4552 Warehouse clubs, supercenters, and ...",4520,General Merchandise Stores,-0.007219,0.016477
2,2024,5221,Depository credit intermediation,5220,Credit Intermediation and Related Activities (...,1.888684,0.014642
3,2024,4561,NAICS 4561 Health and personal care retailers,4461,Health and Personal Care Stores,0.325725,0.009376
4,2024,4551,NAICS 4551 Department stores,4520,General Merchandise Stores,-0.007219,0.008311
5,2024,4581,NAICS 4581 Clothing and clothing accessories r...,4481,Clothing Stores,0.226838,0.007218
6,2024,5313,Activities related to real estate,5310,Real Estate,0.523835,0.007132
7,2024,5311,Lessors of real estate,5310,Real Estate,0.523835,0.005414
8,2024,5132,NAICS 5132 Software publishers,5112,Software Publishers,1.966309,0.005340
9,2024,4571,NAICS 4571 Gasoline stations,4471,Gasoline Stations,-0.310823,0.005331


,year,naics_code,naics_title,sector_weight
0,2024,4841,General freight trucking,0.008670
1,2024,4244,Grocery and related product wholesalers,0.006804
2,2024,4842,Specialized freight trucking,0.003620
3,2024,4491,NAICS 4491 Furniture and home furnishings reta...,0.003371
4,2024,4236,Electric goods merchant wholesalers,0.003080
5,2024,5231,Securities and commodity contracts brokerage,0.003034
6,2024,5312,Offices of real estate agents and brokers,0.002944
7,2024,3323,Architectural and structural metals mfg.,0.002933
8,2024,4237,Hardware and plumbing merchant wholesalers,0.002600
9,2024,4239,Misc. durable goods merchant wholesalers,0.002582


## 3. Accepted Suggested Joins

Load the accepted suggested joins as a distinct reviewed layer that sits on top of the raw baseline.


In [14]:
# Read the first-pass recommendation file and keep only the rows we explicitly accept.
# These rows are still notebook-owned review decisions, but they are distinct from the raw match layer.
naics_recommendations = pd.read_csv(NAICS_RECOMMENDATIONS_PATH, dtype=str)
naics_recommendations_accepted = naics_recommendations.loc[
    naics_recommendations["recommend_match"].astype(str).str.lower() == "true"
].copy()

display(naics_recommendations_accepted.head(20))
display(pd.DataFrame([
    {"metric": "Accepted suggested join rows", "value": len(naics_recommendations_accepted)},
    {"metric": "Accepted suggested join codes", "value": naics_recommendations_accepted["our_code"].nunique()},
]))


,our_code,our_name,our_weight,our_share_of_total,recommended_felten_code,recommended_felten_name,recommended_score,recommended_felten_score,recommend_match,confidence,rationale
0,5324,Machinery and equipment rental and leasing,173283.0,0.0015022189334877555,5321,Automotive Equipment Rental and Leasing,0.7901234567901234,-0.1278947,True,high,Same 3-digit NAICS family with strong title al...
1,3273,Cement and concrete product manufacturing,141627.0,0.0012277878435453585,3270,Nonmetallic Mineral Product Manufacturing,0.6341463414634146,-0.9656745,True,medium,Same 3-digit NAICS family with moderate title ...
2,3371,Household and institutional furniture mfg.,128485.0,0.0011138576759934572,3379,Other Furniture Related Product Manufacturing,0.5416666666666666,-0.5724769,True,medium,Same 3-digit NAICS family with moderate title ...
3,5192,"NAICS 5192 Web search portals, libraries, arch...",124185.0,0.0010765802661263765,5191,Other Information Services,0.5531914893617021,1.749664,True,medium,Same 3-digit NAICS family with moderate title ...
4,3251,Basic chemical manufacturing,98017.0,0.000849725554172477,3254,Pharmaceutical and Medicine Manufacturing,0.6086956521739131,0.4637086,True,medium,Same 3-digit NAICS family with moderate title ...
5,3256,"Soap, cleaning compound, and toiletry mfg.",70614.0,0.0006121644233381484,3254,Pharmaceutical and Medicine Manufacturing,0.5333333333333333,0.4637086,True,medium,Same 3-digit NAICS family with moderate title ...
6,3372,Office furniture and fixtures manufacturing,49034.0,0.00042508384079591535,3379,Other Furniture Related Product Manufacturing,0.7045454545454546,-0.5724769,True,medium,Same 3-digit NAICS family with moderate title ...
7,3255,"Paint, coating, and adhesive manufacturing",41309.0,0.0003581145405114506,3254,Pharmaceutical and Medicine Manufacturing,0.6666666666666666,0.4637086,True,medium,Same 3-digit NAICS family with moderate title ...
8,3279,Other nonmetallic mineral products,39312.0,0.00034080221783597145,3270,Nonmetallic Mineral Product Manufacturing,0.72,-0.9656745,True,medium,Same 3-digit NAICS family with moderate title ...
9,3259,Other chemical product and preparation mfg.,36504.0,0.0003164592022762592,3254,Pharmaceutical and Medicine Manufacturing,0.5806451612903226,0.4637086,True,medium,Same 3-digit NAICS family with moderate title ...


,metric,value
0,Accepted suggested join rows,16
1,Accepted suggested join codes,16


## 4. Combined Notebook Crosswalk

Combine the raw matched rows and the accepted suggested joins into one notebook-owned intermediate crosswalk.


In [15]:
# Build one notebook-owned NAICS crosswalk for review.
# This step is intentionally simple:
# 1. keep the raw normalized matches that already resolve to a Felten score
# 2. layer the accepted suggested joins on top
# 3. anchor the result back to the full live NAICS code universe so step 5 can review the remaining gaps

naics_latest_weights = (
    naics_reference_yearly.loc[naics_reference_yearly["year"] == latest_year, ["naics_code", "sector_employment", "sector_weight"]]
    .drop_duplicates(subset=["naics_code"])
    .rename(columns={"sector_employment": "latest_year_employment", "sector_weight": "latest_year_weight"})
)

naics_raw_matched_rows = (
    naics_raw_status.loc[
        naics_raw_status["raw_normalized_match_flag"],
        [
            "naics_code",
            "naics_title_ours",
            "normalized_naics_code",
            "felten_naics_title_normalized",
            "aiie_score_normalized",
            "raw_exact_match_flag",
        ],
    ]
    .drop_duplicates(subset=["naics_code"], keep="first")
    .rename(
        columns={
            "normalized_naics_code": "selected_felten_code",
            "felten_naics_title_normalized": "selected_felten_name",
            "aiie_score_normalized": "aiie_score",
        }
    )
)
naics_raw_matched_rows["match_basis"] = naics_raw_matched_rows["raw_exact_match_flag"].map(
    {True: "raw_exact_match", False: "raw_normalized_match"}
)
naics_raw_matched_rows["manual_notes"] = pd.NA
naics_raw_matched_rows["review_source"] = "notebook_raw_appendix_b"
naics_raw_matched_rows = naics_raw_matched_rows.drop(columns=["raw_exact_match_flag"]).reset_index(drop=True)

naics_accepted_join_rows = (
    naics_recommendations_accepted[[
        "our_code",
        "our_name",
        "recommended_felten_code",
        "recommended_felten_name",
        "recommended_felten_score",
        "rationale",
    ]]
    .rename(
        columns={
            "our_code": "naics_code",
            "our_name": "naics_title_ours",
            "recommended_felten_code": "selected_felten_code",
            "recommended_felten_name": "selected_felten_name",
            "recommended_felten_score": "aiie_score",
            "rationale": "manual_notes",
        }
    )
    .copy()
)
naics_accepted_join_rows["aiie_score"] = pd.to_numeric(naics_accepted_join_rows["aiie_score"], errors="coerce")
naics_accepted_join_rows["match_basis"] = "accepted_suggested_join"
naics_accepted_join_rows["review_source"] = "recommended_felten_naics_overrides_initial"

naics_step4_matched_base = pd.concat(
    [naics_raw_matched_rows, naics_accepted_join_rows],
    ignore_index=True,
).drop_duplicates(subset=["naics_code"], keep="last").sort_values("naics_code", kind="mergesort").reset_index(drop=True)

naics_step4_review_base = (
    naics_reference_codes.merge(
        naics_step4_matched_base.drop(columns=["naics_title_ours"]),
        on="naics_code",
        how="left",
        validate="one_to_one",
    )
    .merge(
        naics_latest_weights,
        on="naics_code",
        how="left",
        validate="one_to_one",
    )
    .sort_values(["latest_year_weight", "naics_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)
naics_step4_review_base["step4_match_flag"] = naics_step4_review_base["aiie_score"].notna()

naics_step4_coverage = summarize_naics_coverage(
    naics_reference_yearly[["year", "naics_code", "sector_weight"]]
    .drop_duplicates()
    .merge(
        naics_step4_review_base[["naics_code", "step4_match_flag"]],
        on="naics_code",
        how="left",
        validate="many_to_one",
    )
    .assign(step4_match_flag=lambda df: df["step4_match_flag"].fillna(False)),
    "step4_match_flag",
)

naics_step4_missing_latest = (
    naics_step4_review_base.loc[
        ~naics_step4_review_base["step4_match_flag"],
        ["naics_code", "naics_title_ours", "latest_year_employment", "latest_year_weight"],
    ]
    .sort_values(["latest_year_weight", "naics_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

NAICS_STEP4_REVIEW_BASE_PATH.parent.mkdir(parents=True, exist_ok=True)
naics_step4_review_base.to_csv(NAICS_STEP4_REVIEW_BASE_PATH, index=False)

display(pd.DataFrame([
    {"metric": "Raw matched rows kept", "value": len(naics_raw_matched_rows)},
    {"metric": "Accepted suggested joins kept", "value": len(naics_accepted_join_rows)},
    {"metric": "Step 4 review-base rows", "value": len(naics_step4_review_base)},
    {"metric": "Latest-year unresolved after step 4", "value": len(naics_step4_missing_latest)},
    {"metric": "Step 4 review base output", "value": str(NAICS_STEP4_REVIEW_BASE_PATH.relative_to(REPO_ROOT))},
]))
display(naics_step4_coverage.style.format({"coverage_pct": "{:.1%}", "weighted_coverage_pct": "{:.1%}"}))
display(naics_step4_review_base.head(30))
display(naics_step4_missing_latest.head(50))


,metric,value
0,Raw matched rows kept,251
1,Accepted suggested joins kept,16
2,Step 4 review-base rows,332
3,Latest-year unresolved after step 4,67
4,Step 4 review base output,metro-deep-dive/analysis_program/01_ai_inversi...


,year,coverage_pct,weighted_coverage_pct
0,2010,81.4%,79.8%
1,2011,82.0%,88.9%
2,2012,82.0%,89.0%
3,2013,82.0%,89.6%
4,2014,82.0%,89.7%
5,2015,82.0%,89.8%
6,2016,82.0%,90.0%
7,2017,81.9%,89.8%
8,2018,81.9%,89.8%
9,2019,81.9%,90.0%


,naics_code,naics_title_ours,selected_felten_code,selected_felten_name,aiie_score,match_basis,manual_notes,review_source,latest_year_employment,latest_year_weight,step4_match_flag
0,7225,Restaurants,7225,Restaurants and Other Eating Places,-1.171986,raw_exact_match,NaN,notebook_raw_appendix_b,10781478.0,0.093466,True
1,5613,Employment services,5613,Employment Services,-0.723465,raw_exact_match,NaN,notebook_raw_appendix_b,3202176.0,0.027760,True
2,6241,Individual and family services,6241,Individual and Family Services,1.163092,raw_exact_match,NaN,notebook_raw_appendix_b,3111631.0,0.026975,True
3,6221,General medical and surgical hospitals,6221,General Medical and Surgical Hospitals,0.527496,raw_exact_match,NaN,notebook_raw_appendix_b,3077727.0,0.026681,True
4,6211,Offices of physicians,6211,Offices of Physicians,1.009907,raw_exact_match,NaN,notebook_raw_appendix_b,2886021.0,0.025019,True
5,4451,Grocery stores,4450,Food and Beverage Stores (4451 and 4452 only),-0.406696,raw_normalized_match,NaN,notebook_raw_appendix_b,2658001.0,0.023043,True
6,2382,Building equipment contractors,2382,Building Equipment Contractors,-0.832079,raw_exact_match,NaN,notebook_raw_appendix_b,2513189.0,0.021787,True
7,5511,Management of companies and enterprises,5511,Management of Companies and Enterprises,1.698710,raw_exact_match,NaN,notebook_raw_appendix_b,2476752.0,0.021471,True
8,5415,Computer systems design and related services,5415,Computer Systems Design and Related Services,1.860226,raw_exact_match,NaN,notebook_raw_appendix_b,2388342.0,0.020705,True
9,5617,Services to buildings and dwellings,5617,Services to Buildings and Dwellings,-1.998597,raw_exact_match,NaN,notebook_raw_appendix_b,2263587.0,0.019623,True


,naics_code,naics_title_ours,latest_year_employment,latest_year_weight
0,4841,General freight trucking,1000083.0,0.008670
1,4244,Grocery and related product wholesalers,784814.0,0.006804
2,4842,Specialized freight trucking,417587.0,0.003620
3,4491,NAICS 4491 Furniture and home furnishings reta...,388899.0,0.003371
4,4236,Electric goods merchant wholesalers,355269.0,0.003080
5,5231,Securities and commodity contracts brokerage,349932.0,0.003034
6,5312,Offices of real estate agents and brokers,339603.0,0.002944
7,3323,Architectural and structural metals mfg.,338376.0,0.002933
8,4237,Hardware and plumbing merchant wholesalers,299905.0,0.002600
9,4239,Misc. durable goods merchant wholesalers,297854.0,0.002582


## 5. Manual Review

Review the remaining unresolved NAICS codes and add notebook manual locks after inspection.


In [20]:
# QA helper for manual NAICS review.
# Change REVIEW_NAICS_CODE to any live NAICS code you want to inspect.
# This shows:
# 1. the target live code from our step 4 review base
# 2. the live latest-year rows in the same 3-digit family
# 3. the Felten Appendix B rows in the same 3-digit family
# 4. the Felten Appendix B rows in the same 2-digit family

REVIEW_NAICS_CODE = "4491"

review_prefix_3 = str(REVIEW_NAICS_CODE)[:3]
review_prefix_2 = str(REVIEW_NAICS_CODE)[:2]

review_live_code = (
    naics_step4_review_base.loc[naics_step4_review_base["naics_code"] == REVIEW_NAICS_CODE]
    .reset_index(drop=True)
)

review_live_family_3 = (
    naics_step4_review_base.loc[
        naics_step4_review_base["naics_code"].astype(str).str[:3] == review_prefix_3,
        [
            "naics_code",
            "naics_title_ours",
            "latest_year_employment",
            "latest_year_weight",
            "selected_felten_code",
            "selected_felten_name",
            "aiie_score",
            "step4_match_flag",
        ],
    ]
    .sort_values(["latest_year_weight", "naics_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

review_family_appendix_3 = (
    appendix_b.loc[
        appendix_b["felten_naics_code"].astype(str).str[:3] == review_prefix_3,
        ["felten_naics_code", "felten_naics_title", "aiie_score"],
    ]
    .drop_duplicates()
    .sort_values(["felten_naics_code", "felten_naics_title"], kind="mergesort")
    .reset_index(drop=True)
)

review_family_appendix_2 = (
    appendix_b.loc[
        appendix_b["felten_naics_code"].astype(str).str[:2] == review_prefix_2,
        ["felten_naics_code", "felten_naics_title", "aiie_score"],
    ]
    .drop_duplicates()
    .sort_values(["felten_naics_code", "felten_naics_title"], kind="mergesort")
    .reset_index(drop=True)
)

display(pd.DataFrame([
    {"review_naics_code": REVIEW_NAICS_CODE, "family_scope": "3-digit", "prefix": review_prefix_3},
    {"review_naics_code": REVIEW_NAICS_CODE, "family_scope": "2-digit", "prefix": review_prefix_2},
]))

display(review_live_code)
display(review_live_family_3)
display(review_family_appendix_3)
display(review_family_appendix_2)

,review_naics_code,family_scope,prefix
0,4491,3-digit,449
1,4491,2-digit,44


,naics_code,naics_title_ours,selected_felten_code,selected_felten_name,aiie_score,match_basis,manual_notes,review_source,latest_year_employment,latest_year_weight,step4_match_flag
0,4491,NAICS 4491 Furniture and home furnishings reta...,NaN,NaN,NaN,NaN,NaN,NaN,388899.0,0.003371,False


,naics_code,naics_title_ours,latest_year_employment,latest_year_weight,selected_felten_code,selected_felten_name,aiie_score,step4_match_flag
0,4491,NAICS 4491 Furniture and home furnishings reta...,388899.0,0.003371,NaN,NaN,NaN,False
1,4492,NAICS 4492 Electronics and appliance retailers,378136.0,0.003278,4431,Electronics and Appliance Stores,0.361902,True


,felten_naics_code,felten_naics_title,aiie_score


,felten_naics_code,felten_naics_title,aiie_score
0,4411,Automobile Dealers,-0.166889
1,4412,Other Motor Vehicle Dealers,-0.130099
2,4413,"Automotive Parts, Accessories, and Tire Stores",-0.687185
3,4421,Furniture Stores,-0.011229
4,4422,Home Furnishings Stores,0.049807
5,4431,Electronics and Appliance Stores,0.361902
6,4441,Building Material and Supplies Dealers,0.104536
7,4442,Lawn and Garden Equipment and Supplies Stores,-0.464164
8,4450,Food and Beverage Stores (4451 and 4452 only),-0.406696
9,4453,"Beer, Wine, and Liquor Stores",-0.018580


In [22]:
# These are the notebook-level NAICS manual lock decisions we make after reviewing the remaining step 4 gaps.
# The logic here follows a few simple rules:
# 1. if the live code clearly rolls into a Felten aggregate family, use the aggregate code
# 2. if a reviewed proxy is the best available concept match, use that proxy
# 3. leave structurally unmatched concepts out of this table rather than forcing a weak join
naics_notebook_locked_overrides = pd.DataFrame([
    {"naics_code": "4841", "selected_felten_code": "4840", "manual_notes": "Notebook lock: roll current trucking code into Felten's aggregate truck transportation family."},
    {"naics_code": "4244", "selected_felten_code": "4240", "manual_notes": "Notebook lock: roll grocery-related wholesaling into Felten's aggregate nondurable merchant wholesaler family."},
    {"naics_code": "4842", "selected_felten_code": "4840", "manual_notes": "Notebook lock: roll current trucking code into Felten's aggregate truck transportation family."},
    {"naics_code": "4491", "selected_felten_code": "4422", "manual_notes": "Notebook lock: reviewed retail proxy to the closest furniture and home-furnishings concept in Felten."},
    {"naics_code": "4236", "selected_felten_code": "4230", "manual_notes": "Notebook lock: roll durable-goods wholesaling into Felten's aggregate durable merchant wholesaler family."},
    {"naics_code": "5231", "selected_felten_code": "5230", "manual_notes": "Notebook lock: roll brokerage into Felten's aggregate securities and other financial investment family."},
    {"naics_code": "5312", "selected_felten_code": "5310", "manual_notes": "Notebook lock: roll real-estate brokerage into Felten's aggregate real-estate family."},
    {"naics_code": "3323", "selected_felten_code": "3320", "manual_notes": "Notebook lock: roll current fabricated-metals code into Felten's aggregate 3320 family."},
    {"naics_code": "4237", "selected_felten_code": "4230", "manual_notes": "Notebook lock: roll durable-goods wholesaling into Felten's aggregate durable merchant wholesaler family."},
    {"naics_code": "4239", "selected_felten_code": "4230", "manual_notes": "Notebook lock: roll durable-goods wholesaling into Felten's aggregate durable merchant wholesaler family."},
    {"naics_code": "4249", "selected_felten_code": "4240", "manual_notes": "Notebook lock: roll nondurable-goods wholesaling into Felten's aggregate nondurable merchant wholesaler family."},
    {"naics_code": "4233", "selected_felten_code": "4230", "manual_notes": "Notebook lock: roll durable-goods wholesaling into Felten's aggregate durable merchant wholesaler family."},
    {"naics_code": "4242", "selected_felten_code": "4240", "manual_notes": "Notebook lock: roll nondurable-goods wholesaling into Felten's aggregate nondurable merchant wholesaler family."},
    {"naics_code": "5131", "selected_felten_code": "5111", "manual_notes": "Notebook lock: reviewed media-vintage back-map from current 5131 publishing to Felten 5111 publishing."},
    {"naics_code": "3339", "selected_felten_code": "3330", "manual_notes": "Notebook lock: roll machinery manufacturing into Felten's aggregate 3330 family."},
    {"naics_code": "4595", "selected_felten_code": "4520", "manual_notes": "Notebook lock: reviewed retail proxy to Felten's general merchandise stores concept."},
    {"naics_code": "4594", "selected_felten_code": "4539", "manual_notes": "Notebook lock: reviewed retail proxy to Felten's other miscellaneous store retailers concept."},
    {"naics_code": "3329", "selected_felten_code": "3320", "manual_notes": "Notebook lock: roll current fabricated-metals code into Felten's aggregate 3320 family."},
    {"naics_code": "4248", "selected_felten_code": "4240", "manual_notes": "Notebook lock: roll nondurable-goods wholesaling into Felten's aggregate nondurable merchant wholesaler family."},
    {"naics_code": "1113", "selected_felten_code": "1151", "manual_notes": "Notebook lock: use crop-support activities as the closest reviewed proxy for crop production."},
    {"naics_code": "4246", "selected_felten_code": "4240", "manual_notes": "Notebook lock: roll nondurable-goods wholesaling into Felten's aggregate nondurable merchant wholesaler family."},
    {"naics_code": "1114", "selected_felten_code": "1151", "manual_notes": "Notebook lock: use crop-support activities as the closest reviewed proxy for crop production."},
    {"naics_code": "4235", "selected_felten_code": "4230", "manual_notes": "Notebook lock: roll durable-goods wholesaling into Felten's aggregate durable merchant wholesaler family."},
    {"naics_code": "1121", "selected_felten_code": "1152", "manual_notes": "Notebook lock: use animal-support activities as the closest reviewed proxy for animal production."},
    {"naics_code": "5322", "selected_felten_code": "5320", "manual_notes": "Notebook lock: roll consumer-goods rental into Felten's aggregate rental and leasing family."},
    {"naics_code": "4232", "selected_felten_code": "4230", "manual_notes": "Notebook lock: roll durable-goods wholesaling into Felten's aggregate durable merchant wholesaler family."},
    {"naics_code": "4241", "selected_felten_code": "4240", "manual_notes": "Notebook lock: roll nondurable-goods wholesaling into Felten's aggregate nondurable merchant wholesaler family."},
    {"naics_code": "3332", "selected_felten_code": "3330", "manual_notes": "Notebook lock: roll machinery manufacturing into Felten's aggregate 3330 family."},
    {"naics_code": "5161", "selected_felten_code": "5151", "manual_notes": "Notebook lock: reviewed media-vintage back-map from current 5161 broadcasting to Felten 5151 broadcasting."},
    {"naics_code": "3331", "selected_felten_code": "3330", "manual_notes": "Notebook lock: roll machinery manufacturing into Felten's aggregate 3330 family."},
    {"naics_code": "4247", "selected_felten_code": "4240", "manual_notes": "Notebook lock: roll nondurable-goods wholesaling into Felten's aggregate nondurable merchant wholesaler family."},
    {"naics_code": "1112", "selected_felten_code": "1151", "manual_notes": "Notebook lock: use crop-support activities as the closest reviewed proxy for crop production."},
    {"naics_code": "3334", "selected_felten_code": "3330", "manual_notes": "Notebook lock: roll machinery manufacturing into Felten's aggregate 3330 family."},
    {"naics_code": "1119", "selected_felten_code": "1151", "manual_notes": "Notebook lock: use crop-support activities as the closest reviewed proxy for crop production."},
    {"naics_code": "3324", "selected_felten_code": "3320", "manual_notes": "Notebook lock: roll current fabricated-metals code into Felten's aggregate 3320 family."},
    {"naics_code": "1111", "selected_felten_code": "1151", "manual_notes": "Notebook lock: use crop-support activities as the closest reviewed proxy for crop production."},
    {"naics_code": "5178", "selected_felten_code": "5170", "manual_notes": "Notebook lock: roll telecommunications subtype into Felten's aggregate telecommunications family."},
    {"naics_code": "1123", "selected_felten_code": "1152", "manual_notes": "Notebook lock: use animal-support activities as the closest reviewed proxy for animal production."},
    {"naics_code": "5323", "selected_felten_code": "5320", "manual_notes": "Notebook lock: roll general rental centers into Felten's aggregate rental and leasing family."},
    {"naics_code": "3326", "selected_felten_code": "3320", "manual_notes": "Notebook lock: roll current fabricated-metals code into Felten's aggregate 3320 family."},
    {"naics_code": "1122", "selected_felten_code": "1152", "manual_notes": "Notebook lock: use animal-support activities as the closest reviewed proxy for animal production."},
    {"naics_code": "3322", "selected_felten_code": "3320", "manual_notes": "Notebook lock: roll current fabricated-metals code into Felten's aggregate 3320 family."},
    {"naics_code": "1129", "selected_felten_code": "1152", "manual_notes": "Notebook lock: use animal-support activities as the closest reviewed proxy for animal production."},
    {"naics_code": "5232", "selected_felten_code": "5230", "manual_notes": "Notebook lock: roll exchange activity into Felten's aggregate securities and other financial investment family."},
    {"naics_code": "3325", "selected_felten_code": "3320", "manual_notes": "Notebook lock: roll current fabricated-metals code into Felten's aggregate 3320 family."},
    {"naics_code": "5174", "selected_felten_code": "5170", "manual_notes": "Notebook lock: roll telecommunications subtype into Felten's aggregate telecommunications family."},
    {"naics_code": "1125", "selected_felten_code": "1152", "manual_notes": "Notebook lock: use animal-support activities as the closest reviewed proxy for animal production."},
    {"naics_code": "1141", "selected_felten_code": "1152", "manual_notes": "Notebook lock: use animal-support activities as the closest available reviewed proxy for fishing."},
    {"naics_code": "1124", "selected_felten_code": "1152", "manual_notes": "Notebook lock: use animal-support activities as the closest reviewed proxy for animal production."},
    {"naics_code": "1131", "selected_felten_code": "1133", "manual_notes": "Notebook lock: use logging as the closest reviewed forestry proxy."},
    {"naics_code": "1132", "selected_felten_code": "1133", "manual_notes": "Notebook lock: use logging as the closest reviewed forestry proxy."},
    {"naics_code": "4521", "selected_felten_code": "4520", "manual_notes": "Notebook lock: back-map historical department-store code into Felten's aggregate general merchandise family."},
    {"naics_code": "4522", "selected_felten_code": "4520", "manual_notes": "Notebook lock: back-map historical department-store code into Felten's aggregate general merchandise family."},
    {"naics_code": "4523", "selected_felten_code": "4520", "manual_notes": "Notebook lock: back-map historical general-merchandise code into Felten's aggregate general merchandise family."},
    {"naics_code": "4529", "selected_felten_code": "4520", "manual_notes": "Notebook lock: back-map historical general-merchandise code into Felten's aggregate general merchandise family."},
    {"naics_code": "4532", "selected_felten_code": "4530", "manual_notes": "Notebook lock: back-map historical specialty-retail code into Felten's aggregate 4530 family."},
    {"naics_code": "4533", "selected_felten_code": "4530", "manual_notes": "Notebook lock: back-map historical specialty-retail code into Felten's aggregate 4530 family."},
    {"naics_code": "5172", "selected_felten_code": "5170", "manual_notes": "Notebook lock: back-map historical telecommunications subtype into Felten's aggregate telecommunications family."},
    {"naics_code": "5173", "selected_felten_code": "5170", "manual_notes": "Notebook lock: back-map historical telecommunications subtype into Felten's aggregate telecommunications family."},
    {"naics_code": "5179", "selected_felten_code": "5170", "manual_notes": "Notebook lock: back-map historical telecommunications subtype into Felten's aggregate telecommunications family."},
    {"naics_code": "7221", "selected_felten_code": "7225", "manual_notes": "Notebook lock: back-map historical restaurant subtype into Felten's aggregate restaurants family."},
    {"naics_code": "7222", "selected_felten_code": "7225", "manual_notes": "Notebook lock: back-map historical restaurant subtype into Felten's aggregate restaurants family."},
])

if naics_notebook_locked_overrides.empty:
    naics_notebook_locked_overrides = pd.DataFrame(columns=["naics_code", "selected_felten_code", "manual_notes"])

naics_notebook_locked_overrides["review_source"] = "notebook_locked_naics_manual_review"
naics_notebook_locked_overrides["match_basis"] = "manual_override"

display(pd.DataFrame([
    {"input": "Notebook NAICS manual locks", "rows": len(naics_notebook_locked_overrides)},
]))
display(naics_notebook_locked_overrides)

,input,rows
0,Notebook NAICS manual locks,62


,naics_code,selected_felten_code,manual_notes,review_source,match_basis
0,4841,4840,Notebook lock: roll current trucking code into...,notebook_locked_naics_manual_review,manual_override
1,4244,4240,Notebook lock: roll grocery-related wholesalin...,notebook_locked_naics_manual_review,manual_override
2,4842,4840,Notebook lock: roll current trucking code into...,notebook_locked_naics_manual_review,manual_override
3,4491,4422,Notebook lock: reviewed retail proxy to the cl...,notebook_locked_naics_manual_review,manual_override
4,4236,4230,Notebook lock: roll durable-goods wholesaling ...,notebook_locked_naics_manual_review,manual_override
5,5231,5230,Notebook lock: roll brokerage into Felten's ag...,notebook_locked_naics_manual_review,manual_override
6,5312,5310,Notebook lock: roll real-estate brokerage into...,notebook_locked_naics_manual_review,manual_override
7,3323,3320,Notebook lock: roll current fabricated-metals ...,notebook_locked_naics_manual_review,manual_override
8,4237,4230,Notebook lock: roll durable-goods wholesaling ...,notebook_locked_naics_manual_review,manual_override
9,4239,4230,Notebook lock: roll durable-goods wholesaling ...,notebook_locked_naics_manual_review,manual_override


In [21]:
# Build a flat 2-digit-family review export for every latest-year unresolved NAICS code.
# This gives us one row per:
#   unresolved live NAICS code x Felten Appendix B candidate in the same 2-digit family
# so we can review the options more easily outside the notebook.

NAICS_MISSING_2DIGIT_REVIEW_EXPORT_PATH = (
    REPO_ROOT / "metro-deep-dive" / "analysis_program" / "01_ai_inversion" / "outputs" / "naics_missing_2digit_review.csv"
)

naics_missing_latest_for_review = naics_step4_missing_latest.copy()
naics_missing_latest_for_review["naics_prefix_2"] = naics_missing_latest_for_review["naics_code"].astype(str).str[:2]

felten_appendix_b_2digit = appendix_b.copy()
felten_appendix_b_2digit["naics_prefix_2"] = felten_appendix_b_2digit["felten_naics_code"].astype(str).str[:2]

# Pull the broader live 2-digit family too so the export shows which nearby live codes are already matched.
live_naics_2digit = naics_step4_review_base.copy()
live_naics_2digit["naics_prefix_2"] = live_naics_2digit["naics_code"].astype(str).str[:2]

live_naics_2digit_summary = (
    live_naics_2digit.groupby("naics_prefix_2", as_index=False)
    .agg(
        live_codes_in_2digit_family=("naics_code", "nunique"),
        matched_codes_in_2digit_family=("step4_match_flag", "sum"),
    )
)

naics_missing_2digit_review_export = (
    naics_missing_latest_for_review.merge(
        live_naics_2digit_summary,
        on="naics_prefix_2",
        how="left",
        validate="many_to_one",
    )
    .merge(
        felten_appendix_b_2digit[
            ["naics_prefix_2", "felten_naics_code", "felten_naics_title", "aiie_score"]
        ],
        on="naics_prefix_2",
        how="left",
        validate="many_to_many",
    )
    .sort_values(
        ["latest_year_weight", "naics_code", "felten_naics_code"],
        ascending=[False, True, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

NAICS_MISSING_2DIGIT_REVIEW_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
naics_missing_2digit_review_export.to_csv(NAICS_MISSING_2DIGIT_REVIEW_EXPORT_PATH, index=False)

display(pd.DataFrame([
    {"metric": "Latest-year unresolved NAICS codes", "value": len(naics_missing_latest_for_review)},
    {"metric": "2-digit review export rows", "value": len(naics_missing_2digit_review_export)},
    {"metric": "Export path", "value": str(NAICS_MISSING_2DIGIT_REVIEW_EXPORT_PATH.relative_to(REPO_ROOT))},
]))

display(naics_missing_2digit_review_export.head(50))

,metric,value
0,Latest-year unresolved NAICS codes,67
1,2-digit review export rows,812
2,Export path,metro-deep-dive/analysis_program/01_ai_inversi...


,naics_code,naics_title_ours,latest_year_employment,latest_year_weight,naics_prefix_2,live_codes_in_2digit_family,matched_codes_in_2digit_family,felten_naics_code,felten_naics_title,aiie_score
0,4841,General freight trucking,1000083.0,0.008670,48,25,23,4811,Scheduled Air Transportation,-0.455545
1,4841,General freight trucking,1000083.0,0.008670,48,25,23,4812,Nonscheduled Air Transportation,0.113453
2,4841,General freight trucking,1000083.0,0.008670,48,25,23,4821,Rail Transportation,-0.592236
3,4841,General freight trucking,1000083.0,0.008670,48,25,23,4831,"Deep Sea, Coastal, and Great Lakes Water Trans...",-0.328005
4,4841,General freight trucking,1000083.0,0.008670,48,25,23,4832,Inland Water Transportation,-0.871848
5,4841,General freight trucking,1000083.0,0.008670,48,25,23,4840,Truck Transportation,-1.282191
6,4841,General freight trucking,1000083.0,0.008670,48,25,23,4851,Urban Transit Systems,-0.120169
7,4841,General freight trucking,1000083.0,0.008670,48,25,23,4852,Interurban and Rural Bus Transportation,-0.046221
8,4841,General freight trucking,1000083.0,0.008670,48,25,23,4853,Taxi and Limousine Service,1.369857
9,4841,General freight trucking,1000083.0,0.008670,48,25,23,4854,School and Employee Bus Transportation,0.076602


## 6. Final Canonical NAICS Join Table

Combine raw matched rows, accepted suggested joins, and notebook manual locks into the final one-row-per-code table used downstream.


In [23]:
# Build the final canonical NAICS join table from the step 4 review base plus notebook manual locks.
# This final table is the only NAICS join surface we should use downstream once review is complete.

naics_final_crosswalk_base = naics_step4_review_base.merge(
    naics_notebook_locked_overrides[["naics_code", "selected_felten_code", "manual_notes", "review_source", "match_basis"]].rename(
        columns={
            "selected_felten_code": "notebook_selected_felten_code",
            "manual_notes": "notebook_manual_notes",
            "review_source": "notebook_review_source",
            "match_basis": "notebook_match_basis",
        }
    ),
    on="naics_code",
    how="left",
    validate="one_to_one",
)

naics_final_crosswalk_base["selected_felten_code_final"] = naics_final_crosswalk_base["notebook_selected_felten_code"].combine_first(
    naics_final_crosswalk_base["selected_felten_code"]
)
naics_final_crosswalk_base["manual_notes_final"] = naics_final_crosswalk_base["notebook_manual_notes"].combine_first(
    naics_final_crosswalk_base["manual_notes"]
)
naics_final_crosswalk_base["review_source_final"] = naics_final_crosswalk_base["notebook_review_source"].combine_first(
    naics_final_crosswalk_base["review_source"]
)
naics_final_crosswalk_base["match_basis_final"] = naics_final_crosswalk_base["notebook_match_basis"].combine_first(
    naics_final_crosswalk_base["match_basis"]
)

# Resolve the final selected Felten code back to Appendix B so every path uses one consistent source lookup.
felten_naics_lookup = appendix_b_code_lookup.rename(
    columns={
        "felten_naics_code": "selected_felten_code_final",
        "felten_naics_title": "felten_naics_title_final_lookup",
        "aiie_score": "felten_score_final_lookup",
    }
)

naics_final_crosswalk_base = naics_final_crosswalk_base.merge(
    felten_naics_lookup,
    on="selected_felten_code_final",
    how="left",
    validate="many_to_one",
)

naics_final_crosswalk_base["selected_felten_name_final"] = naics_final_crosswalk_base["felten_naics_title_final_lookup"].combine_first(
    naics_final_crosswalk_base["selected_felten_name"]
)
naics_final_crosswalk_base["felten_score_final"] = naics_final_crosswalk_base["felten_score_final_lookup"].combine_first(
    naics_final_crosswalk_base["aiie_score"]
)

naics_felten_join_reference = naics_final_crosswalk_base[[
    "naics_code",
    "naics_title_ours",
    "selected_felten_code_final",
    "selected_felten_name_final",
    "felten_score_final",
    "match_basis_final",
    "manual_notes_final",
    "review_source_final",
]].rename(
    columns={
        "naics_code": "our_naics_code",
        "naics_title_ours": "our_name",
        "selected_felten_code_final": "felten_naics_code",
        "selected_felten_name_final": "felten_naics_name",
        "felten_score_final": "felten_score",
        "match_basis_final": "match_basis",
        "manual_notes_final": "manual_notes",
        "review_source_final": "review_source",
    }
).drop_duplicates(subset=["our_naics_code"], keep="last").sort_values("our_naics_code", kind="mergesort").reset_index(drop=True)

NAICS_FINAL_CROSSWALK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
naics_felten_join_reference.to_csv(NAICS_FINAL_CROSSWALK_OUTPUT_PATH, index=False)

naics_final_status = (
    naics_reference_yearly[["year", "naics_code", "naics_title", "sector_employment", "total_employment", "sector_weight"]]
    .drop_duplicates()
    .merge(
        naics_felten_join_reference[["our_naics_code", "felten_score"]].rename(columns={"our_naics_code": "naics_code"}),
        on="naics_code",
        how="left",
        validate="many_to_one",
    )
)
naics_final_status["final_scored_match_flag"] = naics_final_status["felten_score"].notna()

naics_final_coverage = summarize_naics_coverage(naics_final_status, "final_scored_match_flag")
naics_final_missing_latest = (
    naics_final_status.loc[
        (naics_final_status["year"] == latest_year) & (~naics_final_status["final_scored_match_flag"]),
        ["year", "naics_code", "naics_title", "sector_weight"],
    ]
    .sort_values(["sector_weight", "naics_code"], ascending=[False, True], kind="mergesort")
    .reset_index(drop=True)
)

display(pd.DataFrame([
    {"metric": "Canonical NAICS join rows", "value": len(naics_felten_join_reference)},
    {"metric": "Canonical NAICS rows with score", "value": int(naics_felten_join_reference["felten_score"].notna().sum())},
    {"metric": "Notebook NAICS lock rows applied", "value": int(naics_final_crosswalk_base["notebook_selected_felten_code"].notna().sum())},
    {"metric": "Latest-year unresolved after final locks", "value": len(naics_final_missing_latest)},
    {"metric": "Final join output", "value": str(NAICS_FINAL_CROSSWALK_OUTPUT_PATH.relative_to(REPO_ROOT))},
]))
display(naics_final_coverage.style.format({"coverage_pct": "{:.1%}", "weighted_coverage_pct": "{:.1%}"}))
display(naics_final_missing_latest)
display(naics_felten_join_reference.head(30))


,metric,value
0,Canonical NAICS join rows,332
1,Canonical NAICS rows with score,327
2,Notebook NAICS lock rows applied,62
3,Latest-year unresolved after final locks,5
4,Final join output,metro-deep-dive/analysis_program/01_ai_inversi...


,year,coverage_pct,weighted_coverage_pct
0,2010,98.7%,99.2%
1,2011,98.7%,99.1%
2,2012,98.7%,99.1%
3,2013,98.7%,99.5%
4,2014,98.7%,99.5%
5,2015,98.7%,99.4%
6,2016,98.7%,99.4%
7,2017,98.7%,99.5%
8,2018,98.7%,99.6%
9,2019,98.7%,99.6%


,year,naics_code,naics_title,sector_weight
0,2024,8141,Private households,0.001730
1,2024,9999,Unclassified,0.001558
2,2024,3321,Forging and stamping,0.000464
3,2024,4572,NAICS 4572 Fuel dealers,0.000448
4,2024,1142,Hunting and trapping,0.000002


,our_naics_code,our_name,felten_naics_code,felten_naics_name,felten_score,match_basis,manual_notes,review_source
0,1111,Oilseed and grain farming,1151,Support Activities for Crop Production,-2.165304,manual_override,Notebook lock: use crop-support activities as ...,notebook_locked_naics_manual_review
1,1112,Vegetable and melon farming,1151,Support Activities for Crop Production,-2.165304,manual_override,Notebook lock: use crop-support activities as ...,notebook_locked_naics_manual_review
2,1113,Fruit and tree nut farming,1151,Support Activities for Crop Production,-2.165304,manual_override,Notebook lock: use crop-support activities as ...,notebook_locked_naics_manual_review
3,1114,Greenhouse and nursery production,1151,Support Activities for Crop Production,-2.165304,manual_override,Notebook lock: use crop-support activities as ...,notebook_locked_naics_manual_review
4,1119,Other crop farming,1151,Support Activities for Crop Production,-2.165304,manual_override,Notebook lock: use crop-support activities as ...,notebook_locked_naics_manual_review
5,1121,Cattle ranching and farming,1152,Support Activities for Animal Production,-1.149307,manual_override,Notebook lock: use animal-support activities a...,notebook_locked_naics_manual_review
6,1122,Hog and pig farming,1152,Support Activities for Animal Production,-1.149307,manual_override,Notebook lock: use animal-support activities a...,notebook_locked_naics_manual_review
7,1123,Poultry and egg production,1152,Support Activities for Animal Production,-1.149307,manual_override,Notebook lock: use animal-support activities a...,notebook_locked_naics_manual_review
8,1124,Sheep and goat farming,1152,Support Activities for Animal Production,-1.149307,manual_override,Notebook lock: use animal-support activities a...,notebook_locked_naics_manual_review
9,1125,Aquaculture,1152,Support Activities for Animal Production,-1.149307,manual_override,Notebook lock: use animal-support activities a...,notebook_locked_naics_manual_review
